#  Text-to-Speech (TTS) system Fun-CosyVoice 3.0 and OpenVINO

Fun-CosyVoice 3.0 is an advanced text-to-speech (TTS) system based on large language models (LLM), surpassing its predecessor (CosyVoice 2.0) in content consistency, speaker similarity, and prosody naturalness. It is designed for zero-shot multilingual speech synthesis in the wild.

More details can be found in the original [repository](https://github.com/FunAudioLLM/CosyVoice.git) and [model card](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512)

In this tutorial we consider how to run and optimize Fun-CosyVoice 3.0 using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert and Optimize model](#Convert-and-Optimize-model)
- [Create Inference Pipeline](#Create-Inference-Pipeline)
    - [Select Inference Device](#Select-Inference-Device)
    - [Run Speech Generation](#Run-Speech-Generation)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/janus-multimodal-generation/janus-multimodal-generation.ipynb" />


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/fireredtts2/fireredtts2.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("cosyvoice3-tts.ipynb")

In [2]:
from cmd_helper import clone_repo
from pip_helper import pip_install
import platform
import shutil
import subprocess

pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "nncf",
    "openvino>=2025.4.0",
    "gradio",
    "huggingface_hub",
)

repo_dir = Path("CosyVoice")
revision = "8b54619760fcb78abf5e4637a88e19c1b9ab53c9"
clone_repo("https://github.com/FunAudioLLM/CosyVoice.git", revision)

# Replace CosyVoice requirements.txt with local version
shutil.copy("requirements.txt", repo_dir / "requirements.txt")

pip_install(
    "-q -r",
    str(repo_dir / "requirements.txt"),
)
subprocess.run(
    ["git", "submodule", "update", "--init", "--recursive"],
    cwd=repo_dir,
    check=True
)

if platform.system() == "Darwin":
    pip_install("numpy<2.0")


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Submodule 'third_party/Matcha-TTS' (https://github.com/shivammehta25/Matcha-TTS.git) registered for path 'third_party/Matcha-TTS'
Cloning into '/home2/ethan/intel/openvino_notebooks/notebooks/cosyvoice3-tts/CosyVoice/third_party/Matcha-TTS'...


Submodule path 'third_party/Matcha-TTS': checked out 'dd9105b34bf2be2230f4aa1e4769fb586a3c824e'


## Convert and Optimize model
[back to top ⬆️](#Table-of-contents:)


In [4]:
# Disable torchao to avoid version conflict with torch 2.3.1
import os
os.environ["TRANSFORMERS_NO_TORCHAO"] = "1"

from ov_cosyvoice_helper import convert_cosyvoice
from huggingface_hub import snapshot_download

pt_model_path = "pretrained_models"
snapshot_download(repo_id="FunAudioLLM/Fun-CosyVoice3-0.5B-2512", local_dir=Path(pt_model_path))

model_path = "Fun-CosyVoice3-0.5B-2512-ov"
convert_cosyvoice(pt_model_path, model_path)

2025-12-22 23:21:23,870 DEBUG https://huggingface.co:443 "GET /api/models/FunAudioLLM/Fun-CosyVoice3-0.5B-2512/revision/main HTTP/1.1" 200 1552
2025-12-22 23:21:23,874 DEBUG Attempting to acquire lock 140170200519280 on pretrained_models/.cache/huggingface/download/.gitattributes.lock
2025-12-22 23:21:23,875 DEBUG Lock 140170200519280 acquired on pretrained_models/.cache/huggingface/download/.gitattributes.lock
2025-12-22 23:21:23,876 DEBUG Attempting to release lock 140170200519280 on pretrained_models/.cache/huggingface/download/.gitattributes.lock
2025-12-22 23:21:23,877 DEBUG Lock 140170200519280 released on pretrained_models/.cache/huggingface/download/.gitattributes.lock
2025-12-22 23:21:23,878 DEBUG Attempting to acquire lock 140170200520096 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/config.json.lock
2025-12-22 23:21:23,879 DEBUG Attempting to acquire lock 140170200514960 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/generation_config

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

2025-12-22 23:21:23,883 DEBUG Attempting to acquire lock 140170200527488 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/tokenizer_config.json.lock
2025-12-22 23:21:23,885 DEBUG Lock 140170200519088 acquired on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/merges.txt.lock
2025-12-22 23:21:23,885 DEBUG Attempting to acquire lock 140170200530272 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/model.safetensors.lock
2025-12-22 23:21:23,890 DEBUG Attempting to acquire lock 140171899794288 on pretrained_models/.cache/huggingface/download/asset/dingding.png.lock
2025-12-22 23:21:23,891 DEBUG Attempting to acquire lock 140170712390320 on pretrained_models/.cache/huggingface/download/CosyVoice-BlankEN/vocab.json.lock
2025-12-22 23:21:23,892 DEBUG Attempting to acquire lock 140170712390176 on pretrained_models/.cache/huggingface/download/README.md.lock
2025-12-22 23:21:23,892 DEBUG Attempting to release lock 140170200514960 on pretrained_m

✅ pretrained_models model already converted. You can find results in Fun-CosyVoice3-0.5B-2512-ov


PosixPath('Fun-CosyVoice3-0.5B-2512-ov')

## Create Inference Pipeline
[back to top ⬆️](#Table-of-contents:)

### Select Inference Device
[back to top ⬆️](#Table-of-contents:)

In [5]:
from notebook_utils import device_widget

llm_device = device_widget("CPU", exclude=["AUTO"])

llm_device

Dropdown(description='Device:', options=('CPU', 'GPU'), value='CPU')

In [ ]:
from notebook_utils import device_widget

flow_device = device_widget("CPU", exclude=["NPU"])

flow_device

In [14]:
npu_ov_config = {
            "NPU_USE_NPUW" : "YES",
            "NPUW_LLM" : "YES",
            "NPUW_LLM_SHARED_HEAD" : "False",
            "NPUW_ONLINE_PIPELINE" : "NONE",
        }

In [ ]:
import sys
sys.path.append('CosyVoice/third_party/Matcha-TTS')
from ov_cosyvoice_helper import OVCosyVoice3


# Initialize OpenVINO CosyVoice3
# model_dir: OpenVINO model directory containing all converted models and dependency files
# If dependency files are not in ov_model_dir, provide original model_dir as second argument
cosyvoice = OVCosyVoice3(
    model_dir=model_path,
    device='CPU',  # Default device for all models
    llm_device=llm_device.value,      # LLM model device (defaults to device)
    flow_device=flow_device.value,     # Flow model device (defaults to device)
    hift_device='CPU',     # HiFT model device (defaults to device)
    frontend_device='CPU', # Frontend model device (defaults to device)
    npu_ov_config=npu_ov_config,
)

⌛ Loading OpenVINO frontend models on CPU...
⌛ Loading OpenVINO campplus model from Fun-CosyVoice3-0.5B-2512-ov/campplus.onnx...
✅ Campplus model loaded
⌛ Loading OpenVINO speech tokenizer model from Fun-CosyVoice3-0.5B-2512-ov/speech_tokenizer_v3.onnx...


2025-12-22 23:25:22,590 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


✅ Speech tokenizer model loaded
failed to import ttsfrd, use wetext instead


2025-12-22 23:25:26,769 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 222
2025-12-22 23:25:27,011 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None
2025-12-22 23:25:27,213 DEBUG Starting new HTTPS connection (1): www.modelscope.cn:443


2025-12-22 23:25:29,056 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/revisions HTTP/1.1" 200 222
2025-12-22 23:25:29,775 DEBUG https://www.modelscope.cn:443 "GET /api/v1/models/pengzhendong/wetext/repo/files?Revision=master&Recursive=True HTTP/1.1" 200 None


✅ OpenVINO frontend loaded on CPU
⌛ Loading OpenVINO LLM models on CPU...
⌛ Loading OpenVINO models...
✅ Text embeddings model loaded
✅ Speech embeddings model loaded
✅ LLM model loaded
✅ OpenVINO LLM loaded on CPU
⌛ Loading OpenVINO Flow model on CPU...
⌛ Loading OpenVINO Flow embeddings model from Fun-CosyVoice3-0.5B-2512-ov/openvino_flow_embeddings_model.xml...
✅ Flow embeddings model loaded
⌛ Loading OpenVINO Flow estimator model from Fun-CosyVoice3-0.5B-2512-ov/openvino_flow_estimator_model.xml...
✅ Flow estimator model loaded
✅ OpenVINO Flow loaded on CPU
⌛ Loading OpenVINO HiFT model on CPU...
⌛ Loading OpenVINO HiFT model from Fun-CosyVoice3-0.5B-2512-ov/openvino_hift_model.xml...
✅ HiFT model loaded
✅ OpenVINO HiFT loaded on CPU


### Run zero_shot inference
[back to top ⬆️](#Table-of-contents:)

In [16]:
import torchaudio
import IPython


for i, j in enumerate(cosyvoice.inference_zero_shot(
    '八百标兵奔北坡，北坡炮兵并排跑，炮兵怕把标兵碰，标兵怕碰炮兵炮。',
    'You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_zero_shot_{i}.wav")
    display(IPython.display.Audio(f"ov_zero_shot_{i}.wav"))


  0%|                                                                                                                                                                                                | 0/1 [00:00<?, ?it/s]2025-12-22 23:25:32,419 INFO synthesis text 八百标兵奔北坡，北坡炮兵并排跑，炮兵怕把标兵碰，标兵怕碰炮兵炮。
2025-12-22 23:25:37,679 INFO yield speech len 11.72, rtf 0.44876762217629074


Saved ov_zero_shot_0.wav


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:05<00:00,  5.50s/it]


### Run cross_lingual inference with fine-grained control
[back to top ⬆️](#Table-of-contents:)

In [17]:
for i, j in enumerate(cosyvoice.inference_cross_lingual(
    'You are a helpful assistant.<|endofprompt|>[breath]因为他们那一辈人[breath]在乡里面住的要习惯一点，[breath]邻居都很活络，[breath]嗯，都很熟悉。[breath]',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_fine_grained_control_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_fine_grained_control_{i}.wav")
    display(IPython.display.Audio(f"ov_fine_grained_control_{i}.wav"))

  0%|                                                                                                                                                                                                | 0/1 [00:00<?, ?it/s]2025-12-22 23:25:37,790 INFO synthesis text You are a helpful assistant.<|endofprompt|>[breath]因为他们那一辈人[breath]在乡里面住的要习惯一点，[breath]邻居都很活络，[breath]嗯，都很熟悉。[breath]
2025-12-22 23:25:42,107 INFO yield speech len 10.12, rtf 0.42660175104857434


Saved ov_fine_grained_control_0.wav


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.42s/it]


### Run instruct2 inference (Cantonese)
[back to top ⬆️](#Table-of-contents:)

In [18]:
for i, j in enumerate(cosyvoice.inference_instruct2(
    '好少咯，一般系放嗰啲国庆啊，中秋嗰啲可能会咯。',
    'You are a helpful assistant. 请用广东话表达。<|endofprompt|>',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_instruct_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_instruct_{i}.wav")
    display(IPython.display.Audio(f"ov_instruct_{i}.wav"))

  0%|                                                                                                                                                                                                | 0/1 [00:00<?, ?it/s]2025-12-22 23:25:42,189 INFO synthesis text 好少咯，一般系放嗰啲国庆啊，中秋嗰啲可能会咯。
2025-12-22 23:25:44,666 INFO yield speech len 5.44, rtf 0.455247304018806


Saved ov_instruct_0.wav


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.55s/it]


### Run instruct2 inference (fast speed)
[back to top ⬆️](#Table-of-contents:)

In [19]:
for i, j in enumerate(cosyvoice.inference_instruct2(
    '收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。',
    'You are a helpful assistant. 请用尽可能快地语速说一句话。<|endofprompt|>',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_instruct_fast_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_instruct_fast_{i}.wav")
    display(IPython.display.Audio(f"ov_instruct_fast_{i}.wav"))

  0%|                                                                                                                                                                                                | 0/1 [00:00<?, ?it/s]2025-12-22 23:25:44,740 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-12-22 23:25:47,702 INFO yield speech len 6.76, rtf 0.43814732478215146


Saved ov_instruct_fast_0.wav


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.04s/it]


### Run hotfix inference (fast speed)
[back to top ⬆️](#Table-of-contents:)

In [20]:
for i, j in enumerate(cosyvoice.inference_zero_shot(
    '高管也通过电话、短信、微信等方式对报道[j][ǐ]予好评。',
    'You are a helpful assistant.<|endofprompt|>希望你以后能够做的比我还好呦。',
    './CosyVoice/asset/zero_shot_prompt.wav',
    stream=False
)):
    torchaudio.save('ov_hotfix_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(f"Saved ov_hotfix_{i}.wav")
    display(IPython.display.Audio(f"ov_hotfix_{i}.wav"))

  0%|                                                                                                                                                                                                | 0/1 [00:00<?, ?it/s]2025-12-22 23:25:47,770 INFO synthesis text 高管也通过电话、短信、微信等方式对报道[j][ǐ]予好评。
2025-12-22 23:25:50,416 INFO yield speech len 5.8, rtf 0.45624112260752714


Saved ov_hotfix_0.wav


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.70s/it]


## Interactive demo
[back to top ⬆️](#Table-of-contents:)